# 03 — Select 8,000 HVGs for the scVI branch

This notebook preserves the scientific design of the original analysis:

- input: the metadata-standardized 351,427 × 15,176 full-gene object;
- method: Scanpy/Seurat-v3 highly-variable-gene selection;
- `n_top_genes = 8000`;
- `batch_key = "core_dataset"`;
- raw counts are used;
- the pre-scVI output stores raw counts in `.X` only and intentionally does
  **not** duplicate them into `layers["counts"]`.

## Memory-safe implementation

The full `layers["counts"]` matrix contains ~740 million nonzero entries and
cannot be loaded twice on a 16 GB machine. This version therefore:

1. reads `obs` and `var` without loading either large expression matrix;
2. streams CSR count rows directly from HDF5;
3. reproduces the Seurat-v3 HVG statistics in two streaming passes;
4. uses the same LOESS fit, clipped-count normalized variance, batch-aware
   ranking, and tie-breaking used by `scanpy.pp.highly_variable_genes`;
5. writes the 8,000-gene output in row chunks on the internal NVMe scratch
   drive (default `E:`), then performs an on-disk concatenation;
6. copies each completed output to the repository on `G:` only once.

No full 351k × 15k or 351k × 8k sparse matrix is materialized in RAM.


In [1]:
# =========================
# 0) Imports, paths, and configuration
# =========================

import gc
import hashlib
import inspect
import json
import math
import os
import shutil
import subprocess
import sys
import time
import warnings
from copy import deepcopy
from pathlib import Path

# Reduce accidental CPU thread over-subscription and memory pressure.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")

import anndata as ad
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

try:
    from anndata.io import read_elem
except ImportError:
    from anndata._io.specs import read_elem

try:
    from skmisc.loess import loess
except ImportError as exc:
    raise ImportError(
        "Seurat-v3 HVG selection requires scikit-misc. "
        "Install the same dependency required by "
        "scanpy.pp.highly_variable_genes(flavor='seurat_v3')."
    ) from exc

warnings.filterwarnings("default")


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
LOGS_DIR = PROJECT_DIR / "logs"
INTEGRATED_DIR = PROCESSED_DIR / "integrated"

for directory in (
    PROCESSED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    INTEGRATED_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


INPUT_FULLGENE_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad"
)
FALLBACK_FULLGENE_H5AD = (
    INTEGRATED_DIR
    / "GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad"
)

HVG_N_TOP = 8000
HVG_BATCH_KEY = "core_dataset"
HVG_SPAN = 0.3
EXPECTED_INPUT_SHAPE = (351_427, 15_176)
EXPECTED_OUTPUT_SHAPE = (351_427, 8_000)

OUTPUT_HVG_H5AD = (
    INTEGRATED_DIR
    / f"GBM_core_4datasets_HVG{HVG_N_TOP}_pre_scVI_lowmem_Xcounts.h5ad"
)
OUTPUT_HVG_ALIAS = (
    INTEGRATED_DIR
    / f"GBM_core_HVG{HVG_N_TOP}_pre_scVI_lowmem_Xcounts.h5ad"
)

HVG_TABLE_TSV = (
    METADATA_DIR
    / f"core_4datasets_HVG{HVG_N_TOP}_table_lowmem.tsv"
)
HVG_SELECTED_GENES_TSV = (
    METADATA_DIR
    / f"core_4datasets_HVG{HVG_N_TOP}_selected_genes_lowmem.tsv"
)
HVG_INPUT_INTEGRITY_TSV = (
    METADATA_DIR
    / f"core_4datasets_HVG{HVG_N_TOP}_input_integrity_lowmem.tsv"
)
HVG_SUMMARY_TSV = (
    METADATA_DIR
    / f"core_4datasets_HVG{HVG_N_TOP}_summary_lowmem.tsv"
)
CELL_COUNTS_TSV = (
    METADATA_DIR
    / f"core_4datasets_HVG{HVG_N_TOP}_cell_counts_by_dataset_condition_lowmem.tsv"
)
HVG_OVERLAP_BY_DATASET_TSV = (
    METADATA_DIR
    / f"core_4datasets_HVG{HVG_N_TOP}_overlap_by_dataset_lowmem.tsv"
)

# Large sequential row reads reduce USB/HDF5 call overhead while keeping RAM
# comfortably below the 16 GB system limit.
ROW_CHUNK_SIZE = int(
    os.environ.get(
        "GLIOMA_HVG_ROW_CHUNK_SIZE",
        "4096",
    )
)

CONCAT_MAX_LOADED_ELEMS = int(
    os.environ.get(
        "GLIOMA_HVG_CONCAT_MAX_LOADED_ELEMS",
        "3000000",
    )
)

CACHE_VERSION = "main-analysis-03-streaming-seurat-v3-v2"

_default_scratch = Path(r"E:\glioma-cnv-scratch\main_analysis_03_hvg8000")
SCRATCH_ROOT = Path(
    os.environ.get(
        "GLIOMA_SCRATCH_ROOT_HVG",
        str(_default_scratch),
    )
).expanduser().resolve()

CHUNK_DIR = SCRATCH_ROOT / "chunks"
STAGE_DIR = SCRATCH_ROOT / "stage"

for directory in (
    SCRATCH_ROOT,
    CHUNK_DIR,
    STAGE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

scratch_free_gib = shutil.disk_usage(SCRATCH_ROOT).free / (1024**3)

if scratch_free_gib < 12:
    raise OSError(
        "The HVG notebook needs at least 12 GiB of free NVMe scratch space. "
        f"Observed {scratch_free_gib:.1f} GiB at {SCRATCH_ROOT}."
    )

print("Python:", sys.version)
print("scanpy:", sc.__version__)
print("anndata:", ad.__version__)
print("HVG_N_TOP:", HVG_N_TOP)
print("HVG_BATCH_KEY:", HVG_BATCH_KEY)
print("ROW_CHUNK_SIZE:", ROW_CHUNK_SIZE)
print("SCRATCH_ROOT:", SCRATCH_ROOT)
print("SCRATCH_FREE_GiB:", round(scratch_free_gib, 1))
print("Primary input exists:", INPUT_FULLGENE_H5AD.exists(), INPUT_FULLGENE_H5AD)
print("Fallback input exists:", FALLBACK_FULLGENE_H5AD.exists(), FALLBACK_FULLGENE_H5AD)
print("Output:", OUTPUT_HVG_H5AD)


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
scanpy: 1.11.5
anndata: 0.12.7
HVG_N_TOP: 8000
HVG_BATCH_KEY: core_dataset
ROW_CHUNK_SIZE: 4096
SCRATCH_ROOT: E:\glioma-cnv-scratch\main_analysis_03_hvg8000
SCRATCH_FREE_GiB: 42.5
Primary input exists: True G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad
Fallback input exists: True G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad
Output: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad


C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_27736\1774088873.py:181: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy:", sc.__version__)
C:\Users\AmirSilco\AppData\Local\Temp\ipykernel_27736\1774088873.py:182: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata:", ad.__version__)


In [2]:
# =========================
# 1) HDF5, CSR, file-publication, and report helpers
# =========================

def remove_path(
    path: Path,
    attempts: int = 12,
    initial_delay_seconds: float = 0.25,
):
    path = Path(path)
    delay = float(initial_delay_seconds)
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            if path.is_dir():
                shutil.rmtree(path)
            elif path.exists():
                path.unlink()
            return

        except PermissionError as error:
            last_error = error
            gc.collect()

            if attempt == attempts:
                break

            print(
                "[Windows path-lock retry]",
                f"attempt {attempt}/{attempts}",
                path.name,
                f"sleep={delay:.2f}s",
            )
            time.sleep(delay)
            delay = min(delay * 1.5, 2.0)

    raise PermissionError(
        "Windows did not release the path lock after "
        f"{attempts} attempts: {path}"
    ) from last_error


def write_tsv_replace(
    frame: pd.DataFrame,
    path: Path,
    index=False,
):
    path = Path(path)
    temporary = path.with_name(f"{path.name}.temporary")
    remove_path(temporary)

    frame.to_csv(
        temporary,
        sep="\t",
        index=index,
    )
    os.replace(temporary, path)

    print("[SAVED]", path, "| shape=", frame.shape)


def decode_h5_attribute(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return value


def stored_matrix_shape(node) -> tuple[int, int]:
    if isinstance(node, h5py.Dataset):
        return tuple(int(value) for value in node.shape)

    if not isinstance(node, h5py.Group):
        raise TypeError(
            f"Unsupported matrix node: {type(node).__name__}"
        )

    shape = node.attrs.get("shape")
    if shape is None:
        raise KeyError(
            f"Sparse matrix group {node.name!r} has no shape attribute."
        )

    return tuple(int(value) for value in shape)


def stored_sparse_encoding(group: h5py.Group) -> str:
    return str(
        decode_h5_attribute(
            group.attrs.get("encoding-type", "")
        )
    )


def require_csr_group(
    group: h5py.Group,
    label: str,
):
    encoding = stored_sparse_encoding(group)

    if encoding != "csr_matrix":
        raise RuntimeError(
            f"{label} must be stored as CSR for the streaming reader; "
            f"observed {encoding!r}."
        )

    missing = {
        "data",
        "indices",
        "indptr",
    } - set(group.keys())

    if missing:
        raise KeyError(
            f"{label} is missing CSR arrays: {sorted(missing)}"
        )


def read_csr_rows(
    group: h5py.Group,
    row_start: int,
    row_end: int,
) -> sp.csr_matrix:
    require_csr_group(group, group.name)

    n_rows, n_cols = stored_matrix_shape(group)

    if not (0 <= row_start <= row_end <= n_rows):
        raise IndexError(
            f"Invalid row interval [{row_start}, {row_end}) "
            f"for matrix with {n_rows} rows."
        )

    global_indptr = np.asarray(
        group["indptr"][row_start : row_end + 1],
        dtype=np.int64,
    )

    data_start = int(global_indptr[0])
    data_end = int(global_indptr[-1])

    local_indptr = global_indptr - data_start

    data = np.asarray(
        group["data"][data_start:data_end]
    )
    indices = np.asarray(
        group["indices"][data_start:data_end],
        dtype=np.int32,
    )

    matrix = sp.csr_matrix(
        (
            data,
            indices,
            local_indptr,
        ),
        shape=(row_end - row_start, n_cols),
        dtype=data.dtype,
    )
    matrix.sort_indices()

    return matrix


def validate_raw_counts(
    matrix: sp.csr_matrix,
    label: str,
):
    values = matrix.data

    if values.size == 0:
        return

    if not np.isfinite(values).all():
        raise ValueError(f"{label} contains non-finite values.")

    if np.any(values < 0):
        raise ValueError(f"{label} contains negative values.")

    if not np.issubdtype(values.dtype, np.integer):
        if not np.equal(values, np.floor(values)).all():
            raise ValueError(
                f"{label} contains non-integer values."
            )


def sparse_column_sum_and_sumsq(
    matrix: sp.csr_matrix,
):
    n_genes = matrix.shape[1]

    values64 = matrix.data.astype(
        np.float64,
        copy=False,
    )

    column_sum = np.bincount(
        matrix.indices,
        weights=values64,
        minlength=n_genes,
    ).astype(np.float64, copy=False)

    squared = np.square(values64)

    column_sumsq = np.bincount(
        matrix.indices,
        weights=squared,
        minlength=n_genes,
    ).astype(np.float64, copy=False)

    return column_sum, column_sumsq


def sample_variance_from_moments(
    count: int,
    column_sum: np.ndarray,
    column_sumsq: np.ndarray,
) -> np.ndarray:
    if count <= 1:
        raise ValueError(
            f"At least two cells are required; observed {count}."
        )

    numerator = (
        column_sumsq
        - np.square(column_sum) / float(count)
    )

    # Integer count data should never produce meaningfully negative variance.
    tiny_negative = (
        (numerator < 0)
        & (numerator > -1e-6)
    )
    numerator[tiny_negative] = 0.0

    if np.any(numerator < 0):
        minimum = float(numerator.min())
        raise FloatingPointError(
            f"Negative variance numerator encountered: {minimum}"
        )

    return numerator / float(count - 1)


def clipped_column_sum_and_sumsq(
    matrix: sp.csr_matrix,
    clip_val: np.ndarray,
):
    n_genes = matrix.shape[1]

    values64 = matrix.data.astype(
        np.float64,
        copy=False,
    )

    clipped = np.minimum(
        values64,
        clip_val[matrix.indices],
    )

    column_sum = np.bincount(
        matrix.indices,
        weights=clipped,
        minlength=n_genes,
    ).astype(np.float64, copy=False)

    column_sumsq = np.bincount(
        matrix.indices,
        weights=np.square(clipped),
        minlength=n_genes,
    ).astype(np.float64, copy=False)

    return column_sum, column_sumsq


def strings_sha256(values) -> str:
    digest = hashlib.sha256()

    for value in values:
        digest.update(str(value).encode("utf-8"))
        digest.update(b"\0")

    return digest.hexdigest()


def concat_on_disk_compatible(
    input_files,
    output_file: Path,
):
    output_file = Path(output_file)
    remove_path(output_file)

    function = ad.experimental.concat_on_disk
    supported = set(
        inspect.signature(function).parameters
    )

    optional = {
        "axis": 0,
        "join": "inner",
        "merge": "same",
        "uns_merge": "same",
        "index_unique": None,
        "max_loaded_elems": CONCAT_MAX_LOADED_ELEMS,
    }
    kwargs = {
        key: value
        for key, value in optional.items()
        if key in supported
    }

    config_path = output_file.with_name(
        f"{output_file.name}.concat_config.json"
    )
    remove_path(config_path)

    payload = {
        "in_files": [
            str(Path(path).resolve())
            for path in input_files
        ],
        "out_file": str(output_file.resolve()),
        "kwargs": kwargs,
    }

    with config_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle)

    child_script = r"""
import inspect
import json
import sys
from pathlib import Path

import anndata as ad

config_path = Path(sys.argv[1])

with config_path.open("r", encoding="utf-8") as handle:
    payload = json.load(handle)

function = ad.experimental.concat_on_disk
supported = set(inspect.signature(function).parameters)
kwargs = {
    key: value
    for key, value in payload["kwargs"].items()
    if key in supported
}

print(
    "[child concat_on_disk]",
    len(payload["in_files"]),
    "files ->",
    payload["out_file"],
)
print("[child kwargs]", kwargs)

function(
    in_files=payload["in_files"],
    out_file=payload["out_file"],
    **kwargs,
)
"""

    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"

    try:
        completed = subprocess.run(
            [
                sys.executable,
                "-c",
                child_script,
                str(config_path),
            ],
            cwd=str(PROJECT_DIR),
            env=environment,
            text=True,
            capture_output=True,
            check=False,
        )

        if completed.stdout:
            print(completed.stdout, end="")

        if completed.returncode != 0:
            if completed.stderr:
                print(completed.stderr, file=sys.stderr, end="")

            remove_path(output_file)

            raise RuntimeError(
                "anndata.experimental.concat_on_disk failed in the "
                f"child process with exit code {completed.returncode}. "
                "No in-memory fallback was used."
            )

        if not output_file.exists():
            raise FileNotFoundError(
                "concat_on_disk returned successfully but did not create "
                f"{output_file}."
            )

    finally:
        remove_path(config_path)

    gc.collect()
    time.sleep(0.25)


def patch_uns_in_child_process(
    target_h5ad: Path,
    uns_mapping: dict,
):
    target_h5ad = Path(target_h5ad).resolve()

    sidecar = (
        STAGE_DIR
        / f"{target_h5ad.stem}.{CACHE_VERSION}.uns_sidecar.h5ad"
    )
    remove_path(sidecar)

    holder = ad.AnnData(
        uns=deepcopy(uns_mapping)
    )
    holder.write_h5ad(
        sidecar,
        compression="lzf",
    )

    del holder
    gc.collect()

    child_script = r"""
import sys
import h5py

target_h5ad = sys.argv[1]
sidecar = sys.argv[2]

with h5py.File(sidecar, "r") as source:
    if "uns" not in source:
        raise RuntimeError("UNS sidecar does not contain /uns.")

    with h5py.File(target_h5ad, "r+") as target:
        if "uns" in target:
            del target["uns"]

        source.copy("uns", target, name="uns")
        target.flush()
"""

    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"

    try:
        completed = subprocess.run(
            [
                sys.executable,
                "-c",
                child_script,
                str(target_h5ad),
                str(sidecar),
            ],
            cwd=str(PROJECT_DIR),
            env=environment,
            text=True,
            capture_output=True,
            check=False,
        )

        if completed.returncode != 0:
            if completed.stderr:
                print(completed.stderr, file=sys.stderr, end="")

            raise RuntimeError(
                "The child-process /uns patch failed with exit code "
                f"{completed.returncode}."
            )

    finally:
        remove_path(sidecar)

    gc.collect()
    time.sleep(0.25)


def publish_file(
    scratch_file: Path,
    durable_file: Path,
):
    scratch_file = Path(scratch_file)
    durable_file = Path(durable_file)
    durable_file.parent.mkdir(parents=True, exist_ok=True)

    temporary = durable_file.with_name(
        f"{durable_file.name}.copying"
    )
    remove_path(temporary)

    print(
        "[PUBLISH sequential copy]",
        scratch_file,
        "->",
        durable_file,
    )

    shutil.copyfile(
        scratch_file,
        temporary,
    )

    if temporary.stat().st_size != scratch_file.stat().st_size:
        remove_path(temporary)
        raise AssertionError(
            "Cross-drive publication size mismatch."
        )

    remove_path(durable_file)
    os.replace(
        temporary,
        durable_file,
    )


def validate_hvg_h5ad(
    path: Path,
    expected_obs_names: pd.Index,
    expected_var_names: pd.Index,
):
    path = Path(path)

    with h5py.File(path, "r") as handle:
        if "X" not in handle:
            raise AssertionError(
                f"{path}: /X is missing."
            )

        observed_shape = stored_matrix_shape(
            handle["X"]
        )

        if observed_shape != EXPECTED_OUTPUT_SHAPE:
            raise AssertionError(
                f"{path}: expected shape {EXPECTED_OUTPUT_SHAPE}, "
                f"observed {observed_shape}."
            )

        if isinstance(handle["X"], h5py.Group):
            encoding = stored_sparse_encoding(
                handle["X"]
            )
        else:
            encoding = "dense"

        if encoding != "csr_matrix":
            raise AssertionError(
                f"{path}: X must be CSR; observed {encoding!r}."
            )

        x_dtype = str(
            handle["X"]["data"].dtype
        )

        if (
            "layers" in handle
            and "counts" in handle["layers"]
        ):
            raise AssertionError(
                f"{path}: low-memory output must not contain "
                "layers['counts']."
            )

        saved_obs = read_elem(handle["obs"])
        saved_var = read_elem(handle["var"])
        saved_uns = (
            read_elem(handle["uns"])
            if "uns" in handle
            else {}
        )

    if not saved_obs.index.equals(expected_obs_names):
        raise AssertionError(
            f"{path}: obs_names changed during output assembly."
        )

    if not saved_var.index.equals(expected_var_names):
        raise AssertionError(
            f"{path}: selected gene order changed during output assembly."
        )

    if "hvg_selection" not in saved_uns:
        raise AssertionError(
            f"{path}: uns['hvg_selection'] is missing."
        )

    if "highly_variable" not in saved_var.columns:
        raise AssertionError(
            f"{path}: var['highly_variable'] is missing."
        )

    if not saved_var["highly_variable"].astype(bool).all():
        raise AssertionError(
            f"{path}: every variable in the 8,000-gene object "
            "must be selected as highly variable."
        )

    return {
        "shape": observed_shape,
        "x_encoding": encoding,
        "x_dtype": x_dtype,
        "obs": saved_obs,
        "var": saved_var,
    }


In [3]:
# =========================
# 2) Resolve input and read metadata only
# =========================

if INPUT_FULLGENE_H5AD.exists():
    FULLGENE_INPUT = INPUT_FULLGENE_H5AD
elif FALLBACK_FULLGENE_H5AD.exists():
    FULLGENE_INPUT = FALLBACK_FULLGENE_H5AD
    print(
        "[WARNING] Metadata-standardized object not found. "
        "Using fallback merged object."
    )
else:
    raise FileNotFoundError(
        "No input full-gene h5ad was found. Expected one of:\n"
        f"1) {INPUT_FULLGENE_H5AD}\n"
        f"2) {FALLBACK_FULLGENE_H5AD}"
    )

with h5py.File(FULLGENE_INPUT, "r") as handle:
    if "X" not in handle:
        raise KeyError("Input H5AD is missing /X.")

    if (
        "layers" not in handle
        or "counts" not in handle["layers"]
    ):
        raise KeyError(
            "Input H5AD is missing /layers/counts."
        )

    if "obs" not in handle or "var" not in handle:
        raise KeyError(
            "Input H5AD is missing /obs or /var."
        )

    input_shape = stored_matrix_shape(handle["X"])
    counts_shape = stored_matrix_shape(
        handle["layers"]["counts"]
    )

    require_csr_group(
        handle["layers"]["counts"],
        "layers/counts",
    )

    obs = read_elem(handle["obs"])
    var_original = read_elem(handle["var"])

    counts_encoding = stored_sparse_encoding(
        handle["layers"]["counts"]
    )
    counts_nnz = int(
        handle["layers"]["counts"]["data"].shape[0]
    )
    counts_dtype = str(
        handle["layers"]["counts"]["data"].dtype
    )

if input_shape != EXPECTED_INPUT_SHAPE:
    raise AssertionError(
        f"Expected full-gene input shape {EXPECTED_INPUT_SHAPE}, "
        f"observed {input_shape}."
    )

if counts_shape != input_shape:
    raise AssertionError(
        f"counts shape {counts_shape} does not match X shape {input_shape}."
    )

if len(obs) != input_shape[0]:
    raise AssertionError("obs row count does not match the matrix.")

if len(var_original) != input_shape[1]:
    raise AssertionError("var row count does not match the matrix.")

if not obs.index.is_unique:
    raise ValueError("Duplicate cell barcodes found.")

if not var_original.index.is_unique:
    raise ValueError("Duplicate gene symbols found.")

required_obs_cols = [
    "core_dataset",
    "dataset_id",
    "geo_id",
    "sample_id",
    "patient_id",
    "condition",
    "scvi_batch",
]

missing_required = [
    column
    for column in required_obs_cols
    if column not in obs.columns
]

if missing_required:
    raise KeyError(
        "Missing required obs columns: "
        + ", ".join(missing_required)
    )

for column in required_obs_cols:
    values = obs[column].astype(str)
    values = values.replace(
        {
            "nan": "unknown",
            "None": "unknown",
            "<NA>": "unknown",
            "": "unknown",
        }
    )
    obs[column] = values

batch_values = obs[HVG_BATCH_KEY].astype(str).to_numpy()
batch_labels = np.unique(batch_values)

if len(batch_labels) != 4:
    raise AssertionError(
        f"Expected four {HVG_BATCH_KEY!r} batches, "
        f"observed {len(batch_labels)}: {batch_labels.tolist()}"
    )

batch_counts = {
    label: int(np.sum(batch_values == label))
    for label in batch_labels
}

print("Input file:", FULLGENE_INPUT)
print("Input shape:", input_shape)
print("counts encoding:", counts_encoding)
print("counts dtype:", counts_dtype)
print("counts nonzero entries:", f"{counts_nnz:,}")
print("batches:", batch_counts)

integrity_rows = [
    {
        "field": "input_fullgene_h5ad",
        "value": str(FULLGENE_INPUT),
    },
    {
        "field": "n_cells",
        "value": int(input_shape[0]),
    },
    {
        "field": "n_genes",
        "value": int(input_shape[1]),
    },
    {
        "field": "has_counts_layer",
        "value": True,
    },
    {
        "field": "counts_encoding",
        "value": counts_encoding,
    },
    {
        "field": "counts_dtype",
        "value": counts_dtype,
    },
    {
        "field": "counts_nonzero_entries",
        "value": counts_nnz,
    },
    {
        "field": "output_hvg_h5ad",
        "value": str(OUTPUT_HVG_H5AD),
    },
    {
        "field": "memory_mode",
        "value": "streaming_direct_hdf5_seurat_v3",
    },
]

for column in required_obs_cols:
    integrity_rows.append(
        {
            "field": column,
            "value": (
                f"n_unique={obs[column].nunique()}; "
                f"n_unknown={(obs[column].astype(str) == 'unknown').sum()}"
            ),
        }
    )

integrity = pd.DataFrame(integrity_rows)
write_tsv_replace(
    integrity,
    HVG_INPUT_INTEGRITY_TSV,
)
display(integrity)


Input file: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad
Input shape: (351427, 15176)
counts encoding: csr_matrix
counts dtype: int32
counts nonzero entries: 739,913,124
batches: {'DS1_GSE103224': 23207, 'DS2_GSE141383': 21391, 'DS3_GSE182109': 218735, 'DS4_GSE278450': 88094}
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\core_4datasets_HVG8000_input_integrity_lowmem.tsv | shape= (16, 2)


,field,value
0,input_fullgene_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,n_cells,351427
2,n_genes,15176
3,has_counts_layer,True
4,counts_encoding,csr_matrix
5,counts_dtype,int32
6,counts_nonzero_entries,739913124
7,output_hvg_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
8,memory_mode,streaming_direct_hdf5_seurat_v3
9,core_dataset,n_unique=4; n_unknown=0


In [4]:
# =========================
# 3) Cell-composition report
# =========================

cell_counts = (
    obs
    .groupby(
        [
            "core_dataset",
            "condition",
            "sample_id",
            "patient_id",
        ],
        observed=True,
    )
    .size()
    .reset_index(name="n_cells")
    .sort_values(
        [
            "core_dataset",
            "condition",
            "sample_id",
            "patient_id",
        ]
    )
)

write_tsv_replace(
    cell_counts,
    CELL_COUNTS_TSV,
)

display(cell_counts.head(20))

print("Cell counts by core_dataset:")
display(
    obs["core_dataset"]
    .value_counts()
    .rename_axis("core_dataset")
    .reset_index(name="n_cells")
)

print("Cell counts by condition:")
display(
    obs["condition"]
    .value_counts()
    .rename_axis("condition")
    .reset_index(name="n_cells")
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\core_4datasets_HVG8000_cell_counts_by_dataset_condition_lowmem.tsv | shape= (83, 5)


,core_dataset,condition,sample_id,patient_id,n_cells
0,DS1_GSE103224,GBM,PJ016,PJ016,2838
1,DS1_GSE103224,GBM,PJ017,PJ017,1237
2,DS1_GSE103224,GBM,PJ018,PJ018,2163
3,DS1_GSE103224,GBM,PJ025,PJ025,5829
4,DS1_GSE103224,GBM,PJ030,PJ030,3048
5,DS1_GSE103224,GBM,PJ032,PJ032,1354
6,DS1_GSE103224,GBM,PJ035,PJ035,3707
7,DS1_GSE103224,GBM,PJ048,PJ048,3031
8,DS2_GSE141383,GBM,PDC001,PDC001,6325
9,DS2_GSE141383,GBM,PJ052,PJ052,1364


Cell counts by core_dataset:


,core_dataset,n_cells
0,DS3_GSE182109,218735
1,DS4_GSE278450,88094
2,DS1_GSE103224,23207
3,DS2_GSE141383,21391


Cell counts by condition:


,condition,n_cells
0,GBM,132692
1,ndGBM,112416
2,rGBM,92832
3,LGG,13487


In [5]:
# =========================
# 4) Streaming pass 1:
#    global and per-batch mean/variance moments
# =========================

n_cells, n_genes_full = input_shape

global_sum = np.zeros(
    n_genes_full,
    dtype=np.float64,
)
global_sumsq = np.zeros(
    n_genes_full,
    dtype=np.float64,
)

batch_sum = {
    label: np.zeros(
        n_genes_full,
        dtype=np.float64,
    )
    for label in batch_labels
}
batch_sumsq = {
    label: np.zeros(
        n_genes_full,
        dtype=np.float64,
    )
    for label in batch_labels
}

maximum_count = 0
observed_nnz = 0

print("Streaming pass 1 over raw counts...")

with h5py.File(FULLGENE_INPUT, "r") as handle:
    counts_group = handle["layers"]["counts"]

    for chunk_index, row_start in enumerate(
        range(0, n_cells, ROW_CHUNK_SIZE)
    ):
        row_end = min(
            row_start + ROW_CHUNK_SIZE,
            n_cells,
        )

        chunk = read_csr_rows(
            counts_group,
            row_start,
            row_end,
        )

        validate_raw_counts(
            chunk,
            f"rows {row_start}:{row_end}",
        )

        if chunk.nnz:
            maximum_count = max(
                maximum_count,
                int(chunk.data.max()),
            )
        observed_nnz += int(chunk.nnz)

        chunk_sum, chunk_sumsq = (
            sparse_column_sum_and_sumsq(chunk)
        )
        global_sum += chunk_sum
        global_sumsq += chunk_sumsq

        labels_chunk = batch_values[
            row_start:row_end
        ]

        for label in np.unique(labels_chunk):
            row_mask = labels_chunk == label

            if row_mask.all():
                batch_chunk = chunk
                owns_batch_chunk = False
            else:
                batch_chunk = chunk[
                    row_mask,
                    :
                ].tocsr()
                owns_batch_chunk = True

            current_sum, current_sumsq = (
                sparse_column_sum_and_sumsq(
                    batch_chunk
                )
            )

            batch_sum[label] += current_sum
            batch_sumsq[label] += current_sumsq

            if owns_batch_chunk:
                del batch_chunk

        if (
            chunk_index == 0
            or (chunk_index + 1) % 20 == 0
            or row_end == n_cells
        ):
            print(
                f"  pass 1 chunk {chunk_index + 1}: "
                f"rows {row_start:,}:{row_end:,}"
            )

        del chunk
        gc.collect()

if observed_nnz != counts_nnz:
    raise AssertionError(
        f"Streaming nnz mismatch: observed {observed_nnz:,}, "
        f"stored {counts_nnz:,}."
    )

global_mean_full = (
    global_sum / float(n_cells)
)
global_var_full = (
    sample_variance_from_moments(
        n_cells,
        global_sum,
        global_sumsq,
    )
)

keep_nonzero_genes = global_sum > 0
n_zero_genes = int(
    (~keep_nonzero_genes).sum()
)

print("Zero-count genes:", n_zero_genes)
print("Maximum raw count:", maximum_count)

if int(keep_nonzero_genes.sum()) < HVG_N_TOP:
    raise ValueError(
        f"Only {int(keep_nonzero_genes.sum())} nonzero genes are available, "
        f"fewer than requested HVG_N_TOP={HVG_N_TOP}."
    )

analysis_positions = np.flatnonzero(
    keep_nonzero_genes
).astype(np.int32)

analysis_var = var_original.iloc[
    analysis_positions
].copy()

analysis_gene_names = pd.Index(
    analysis_var.index.astype(str),
    dtype="object",
)

global_mean = global_mean_full[
    analysis_positions
]
global_var = global_var_full[
    analysis_positions
]

batch_mean = {}
batch_var = {}

for label in batch_labels:
    count = batch_counts[label]

    batch_mean[label] = (
        batch_sum[label][analysis_positions]
        / float(count)
    )

    batch_var[label] = (
        sample_variance_from_moments(
            count,
            batch_sum[label][analysis_positions],
            batch_sumsq[label][analysis_positions],
        )
    )

print(
    "Genes entering Seurat-v3 HVG calculation:",
    len(analysis_gene_names),
)


Streaming pass 1 over raw counts...
  pass 1 chunk 1: rows 0:4,096
  pass 1 chunk 20: rows 77,824:81,920
  pass 1 chunk 40: rows 159,744:163,840
  pass 1 chunk 60: rows 241,664:245,760
  pass 1 chunk 80: rows 323,584:327,680
  pass 1 chunk 86: rows 348,160:351,427
Zero-count genes: 0
Maximum raw count: 26377
Genes entering Seurat-v3 HVG calculation: 15176


In [6]:
# =========================
# 5) Seurat-v3 LOESS fits and streaming clipped-count pass
# =========================

n_analysis_genes = len(analysis_gene_names)

clip_values = {}
regularized_std = {}

for label in batch_labels:
    mean = batch_mean[label]
    variance = batch_var[label]

    estimated_variance = np.zeros(
        n_analysis_genes,
        dtype=np.float64,
    )

    not_constant = variance > 0

    if not_constant.any():
        x = np.log10(
            mean[not_constant]
        )
        y = np.log10(
            variance[not_constant]
        )

        model = loess(
            x,
            y,
            span=HVG_SPAN,
            degree=2,
        )
        model.fit()

        estimated_variance[
            not_constant
        ] = model.outputs.fitted_values

    reg_std = np.sqrt(
        10 ** estimated_variance
    )

    clip_val = (
        reg_std
        * np.sqrt(batch_counts[label])
        + mean
    )

    regularized_std[label] = reg_std
    clip_values[label] = clip_val


clipped_sum = {
    label: np.zeros(
        n_analysis_genes,
        dtype=np.float64,
    )
    for label in batch_labels
}
clipped_sumsq = {
    label: np.zeros(
        n_analysis_genes,
        dtype=np.float64,
    )
    for label in batch_labels
}

print("Streaming pass 2 for Seurat-v3 clipped-count variance...")

with h5py.File(FULLGENE_INPUT, "r") as handle:
    counts_group = handle["layers"]["counts"]

    for chunk_index, row_start in enumerate(
        range(0, n_cells, ROW_CHUNK_SIZE)
    ):
        row_end = min(
            row_start + ROW_CHUNK_SIZE,
            n_cells,
        )

        full_chunk = read_csr_rows(
            counts_group,
            row_start,
            row_end,
        )

        if n_zero_genes:
            chunk = full_chunk[
                :,
                analysis_positions,
            ].tocsr()
            del full_chunk
        else:
            chunk = full_chunk

        labels_chunk = batch_values[
            row_start:row_end
        ]

        for label in np.unique(labels_chunk):
            row_mask = labels_chunk == label

            if row_mask.all():
                batch_chunk = chunk
                owns_batch_chunk = False
            else:
                batch_chunk = chunk[
                    row_mask,
                    :
                ].tocsr()
                owns_batch_chunk = True

            current_sum, current_sumsq = (
                clipped_column_sum_and_sumsq(
                    batch_chunk,
                    clip_values[label],
                )
            )

            clipped_sum[label] += current_sum
            clipped_sumsq[label] += current_sumsq

            if owns_batch_chunk:
                del batch_chunk

        if (
            chunk_index == 0
            or (chunk_index + 1) % 20 == 0
            or row_end == n_cells
        ):
            print(
                f"  pass 2 chunk {chunk_index + 1}: "
                f"rows {row_start:,}:{row_end:,}"
            )

        del chunk
        gc.collect()


norm_gene_vars = []

for label in batch_labels:
    n_obs_batch = batch_counts[label]
    mean = batch_mean[label]
    reg_std = regularized_std[label]

    numerator = (
        n_obs_batch * np.square(mean)
        + clipped_sumsq[label]
        - 2.0 * clipped_sum[label] * mean
    )

    denominator = (
        (n_obs_batch - 1)
        * np.square(reg_std)
    )

    normalized_variance = (
        numerator / denominator
    )

    norm_gene_vars.append(
        normalized_variance.reshape(1, -1)
    )


norm_gene_vars = np.concatenate(
    norm_gene_vars,
    axis=0,
)

# Exact Scanpy/Seurat-v3 batch-aware ranking logic.
ranked_norm_gene_vars = np.argsort(
    np.argsort(
        -norm_gene_vars,
        axis=1,
    ),
    axis=1,
)

ranked_norm_gene_vars = (
    ranked_norm_gene_vars.astype(
        np.float32
    )
)

num_batches_high_var = np.sum(
    (
        ranked_norm_gene_vars
        < HVG_N_TOP
    ).astype(int),
    axis=0,
)

ranked_norm_gene_vars[
    ranked_norm_gene_vars
    >= HVG_N_TOP
] = np.nan

masked_ranked = np.ma.masked_invalid(
    ranked_norm_gene_vars
)

median_ranked = np.ma.median(
    masked_ranked,
    axis=0,
).filled(np.nan)


hvg_metrics = pd.DataFrame(
    index=analysis_gene_names
)

hvg_metrics["means"] = global_mean
hvg_metrics["variances"] = global_var
hvg_metrics[
    "highly_variable_nbatches"
] = num_batches_high_var
hvg_metrics[
    "highly_variable_rank"
] = median_ranked
hvg_metrics[
    "variances_norm"
] = np.mean(
    norm_gene_vars,
    axis=0,
)

sorted_index = (
    hvg_metrics[
        [
            "highly_variable_rank",
            "highly_variable_nbatches",
        ]
    ]
    .sort_values(
        [
            "highly_variable_rank",
            "highly_variable_nbatches",
        ],
        ascending=[True, False],
        na_position="last",
    )
    .index
)

hvg_metrics["highly_variable"] = False
hvg_metrics.loc[
    sorted_index[:HVG_N_TOP],
    "highly_variable",
] = True

n_hvg = int(
    hvg_metrics["highly_variable"].sum()
)

if n_hvg != HVG_N_TOP:
    raise AssertionError(
        f"Expected exactly {HVG_N_TOP} selected HVGs, observed {n_hvg}."
    )

print("[HVG] streaming Seurat-v3 completed successfully.")
print("HVG selected:", n_hvg)

# Reproduce the var columns that Scanpy would add in-place.
var_with_hvg = analysis_var.copy()
for column in (
    "highly_variable",
    "highly_variable_rank",
    "means",
    "variances",
    "variances_norm",
    "highly_variable_nbatches",
):
    var_with_hvg[column] = (
        hvg_metrics[column]
        .reindex(var_with_hvg.index)
        .to_numpy()
    )

hvg_method_used = (
    "seurat_v3_counts_batch_core_dataset_"
    "streaming_exact_formula"
)

hvg_table = var_with_hvg.copy()

if "gene" in hvg_table.columns:
    hvg_table = hvg_table.drop(
        columns=["gene"]
    )

hvg_table.insert(
    0,
    "gene",
    hvg_table.index.astype(str),
)
hvg_table["selected_hvg"] = (
    hvg_table["highly_variable"]
    .astype(bool)
)
hvg_table["hvg_method_used"] = (
    hvg_method_used
)
hvg_table["hvg_batch_key"] = (
    HVG_BATCH_KEY
)

write_tsv_replace(
    hvg_table,
    HVG_TABLE_TSV,
)

selected = hvg_table[
    hvg_table["selected_hvg"]
].copy()

sort_columns = [
    column
    for column in (
        "highly_variable_rank",
        "highly_variable_nbatches",
        "means",
    )
    if column in selected.columns
]

ascending_map = {
    "highly_variable_rank": True,
    "highly_variable_nbatches": False,
    "means": False,
}

if sort_columns:
    selected = selected.sort_values(
        sort_columns,
        ascending=[
            ascending_map[column]
            for column in sort_columns
        ],
    )

write_tsv_replace(
    selected,
    HVG_SELECTED_GENES_TSV,
)

display(selected.head(20))


Streaming pass 2 for Seurat-v3 clipped-count variance...
  pass 2 chunk 1: rows 0:4,096
  pass 2 chunk 20: rows 77,824:81,920
  pass 2 chunk 40: rows 159,744:163,840
  pass 2 chunk 60: rows 241,664:245,760
  pass 2 chunk 80: rows 323,584:327,680
  pass 2 chunk 86: rows 348,160:351,427
[HVG] streaming Seurat-v3 completed successfully.
HVG selected: 8000
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\core_4datasets_HVG8000_table_lowmem.tsv | shape= (15176, 10)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\core_4datasets_HVG8000_selected_genes_lowmem.tsv | shape= (8000, 10)


,gene,highly_variable,highly_variable_rank,means,variances,variances_norm,highly_variable_nbatches,selected_hvg,hvg_method_used,hvg_batch_key
SAA1,SAA1,True,2.0,0.287388,78.309251,39.090897,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
PLA2G2A,PLA2G2A,True,2.5,0.312973,68.586218,42.580223,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
LTF,LTF,True,3.0,0.282218,62.118281,42.521489,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
CXCL10,CXCL10,True,4.0,0.297806,81.952239,34.745107,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
SAA2,SAA2,True,11.0,0.060405,6.622753,20.656876,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
PI3,PI3,True,12.0,0.128126,19.749944,27.328736,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
MGP,MGP,True,13.0,0.877255,145.659738,23.647433,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
CHI3L2,CHI3L2,True,14.5,0.248524,8.534043,15.224936,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
CLDN5,CLDN5,True,16.0,0.210997,15.137982,19.123605,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset
RGS5,RGS5,True,18.0,0.125155,6.625430,16.127739,4,True,seurat_v3_counts_batch_core_dataset_streaming_...,core_dataset


In [7]:
# =========================
# 6) Build the 8,000-gene AnnData in row chunks on NVMe
# =========================

hvg_mask = (
    var_with_hvg["highly_variable"]
    .astype(bool)
    .to_numpy()
)

selected_gene_positions_within_analysis = (
    np.flatnonzero(hvg_mask)
    .astype(np.int32)
)

selected_full_positions = (
    analysis_positions[
        selected_gene_positions_within_analysis
    ]
)

selected_genes = pd.Index(
    var_with_hvg.index[
        selected_gene_positions_within_analysis
    ].astype(str),
    dtype="object",
)

if len(selected_genes) != HVG_N_TOP:
    raise AssertionError(
        f"Expected {HVG_N_TOP} selected genes, "
        f"observed {len(selected_genes)}."
    )

var_hvg = var_with_hvg.iloc[
    selected_gene_positions_within_analysis
].copy()

# Match the original low-memory notebook: selected genes stay in the
# original full-gene order, not rank order.
if not var_hvg.index.equals(selected_genes):
    raise AssertionError(
        "Selected gene order is inconsistent."
    )

obs_hvg = obs.copy()

for column in (
    "core_dataset",
    "dataset_id",
    "geo_id",
    "study_id",
    "sample_id",
    "patient_id",
    "condition",
    "diagnosis",
    "tumor_status",
    "scvi_batch",
):
    if column in obs_hvg.columns:
        obs_hvg[column] = (
            obs_hvg[column]
            .astype("category")
        )


hvg_uns = {
    "hvg_selection": {
        "source_fullgene_h5ad": str(
            FULLGENE_INPUT
        ),
        "output_hvg_h5ad": str(
            OUTPUT_HVG_H5AD
        ),
        "n_top_genes_requested": int(
            HVG_N_TOP
        ),
        "n_hvg_selected": int(
            len(selected_genes)
        ),
        "hvg_method_used": (
            hvg_method_used
        ),
        "hvg_batch_key": (
            HVG_BATCH_KEY
        ),
        "hvg_span": float(
            HVG_SPAN
        ),
        "x_in_output": "raw counts",
        "counts_layer_in_output": (
            "not stored to avoid duplicating memory; "
            "use scVI layer=None"
        ),
        "memory_mode": (
            "streaming_direct_hdf5_two_pass_"
            "seurat_v3_plus_chunked_output"
        ),
        "selected_gene_order": (
            "original full-gene order"
        ),
        "selected_gene_sha256": (
            strings_sha256(selected_genes)
        ),
        "note": (
            "This is the HVG/scVI branch. "
            "The full-gene object remains the source "
            "for marker DE and CNV."
        ),
        "cache_version": CACHE_VERSION,
    }
}


def write_hvg_chunk(
    matrix: sp.csr_matrix,
    obs_chunk: pd.DataFrame,
    output_path: Path,
):
    output_path = Path(output_path)
    temporary = output_path.with_name(
        f"{output_path.name}.temporary"
    )

    remove_path(temporary)
    remove_path(output_path)

    matrix = matrix.tocsr()
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    matrix.sort_indices()

    chunk = ad.AnnData(
        X=matrix,
        obs=obs_chunk,
        var=var_hvg,
    )

    # Deliberately do NOT create layers["counts"].
    chunk.write_h5ad(
        temporary,
        compression="lzf",
    )

    os.replace(
        temporary,
        output_path,
    )

    del chunk
    gc.collect()


remove_path(CHUNK_DIR)
CHUNK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

chunk_files = []

print(
    "Building 8,000-gene output chunks on NVMe..."
)

with h5py.File(FULLGENE_INPUT, "r") as handle:
    counts_group = handle["layers"]["counts"]

    for chunk_index, row_start in enumerate(
        range(0, n_cells, ROW_CHUNK_SIZE)
    ):
        row_end = min(
            row_start + ROW_CHUNK_SIZE,
            n_cells,
        )

        full_chunk = read_csr_rows(
            counts_group,
            row_start,
            row_end,
        )

        hvg_chunk = full_chunk[
            :,
            selected_full_positions,
        ].tocsr()

        del full_chunk

        chunk_path = (
            CHUNK_DIR
            / f"chunk_{chunk_index:06d}.h5ad"
        )

        write_hvg_chunk(
            hvg_chunk,
            obs_hvg.iloc[
                row_start:row_end
            ].copy(),
            chunk_path,
        )

        chunk_files.append(
            chunk_path
        )

        if (
            chunk_index == 0
            or (chunk_index + 1) % 20 == 0
            or row_end == n_cells
        ):
            print(
                f"  output chunk {chunk_index + 1}: "
                f"rows {row_start:,}:{row_end:,}"
            )

        del hvg_chunk
        gc.collect()


scratch_output = (
    STAGE_DIR
    / f"GBM_core_4datasets_HVG{HVG_N_TOP}.{CACHE_VERSION}.assembled.h5ad"
)

remove_path(scratch_output)

try:
    concat_on_disk_compatible(
        chunk_files,
        scratch_output,
    )

    patch_uns_in_child_process(
        scratch_output,
        hvg_uns,
    )

    scratch_validation = validate_hvg_h5ad(
        scratch_output,
        expected_obs_names=obs_hvg.index,
        expected_var_names=selected_genes,
    )

    print(
        "Scratch output validated:",
        scratch_validation["shape"],
        scratch_validation["x_encoding"],
        scratch_validation["x_dtype"],
    )

    publish_file(
        scratch_output,
        OUTPUT_HVG_H5AD,
    )

    primary_validation = validate_hvg_h5ad(
        OUTPUT_HVG_H5AD,
        expected_obs_names=obs_hvg.index,
        expected_var_names=selected_genes,
    )

    # Preserve the historical compatibility alias. Copy directly from the
    # completed NVMe scratch file so the slow USB drive is only a write target.
    publish_file(
        scratch_output,
        OUTPUT_HVG_ALIAS,
    )

    alias_validation = validate_hvg_h5ad(
        OUTPUT_HVG_ALIAS,
        expected_obs_names=obs_hvg.index,
        expected_var_names=selected_genes,
    )

finally:
    remove_path(scratch_output)

# Successful durable outputs exist; transient chunks are no longer needed.
remove_path(CHUNK_DIR)

print()
print("[SAVED]", OUTPUT_HVG_H5AD)
print("[SAVED alias]", OUTPUT_HVG_ALIAS)
print("Validated shape:", primary_validation["shape"])
print("Validated X encoding:", primary_validation["x_encoding"])
print("Validated X dtype:", primary_validation["x_dtype"])
print("Validated layers: no counts layer")
print("Validated selected genes:", len(selected_genes))


Building 8,000-gene output chunks on NVMe...
  output chunk 1: rows 0:4,096
  output chunk 20: rows 77,824:81,920
  output chunk 40: rows 159,744:163,840
  output chunk 60: rows 241,664:245,760
  output chunk 80: rows 323,584:327,680
  output chunk 86: rows 348,160:351,427
[child concat_on_disk] 86 files -> E:\glioma-cnv-scratch\main_analysis_03_hvg8000\stage\GBM_core_4datasets_HVG8000.main-analysis-03-streaming-seurat-v3-v2.assembled.h5ad
[child kwargs] {'axis': 0, 'join': 'inner', 'merge': 'same', 'uns_merge': 'same', 'index_unique': None, 'max_loaded_elems': 3000000}
Scratch output validated: (351427, 8000) csr_matrix int32
[PUBLISH sequential copy] E:\glioma-cnv-scratch\main_analysis_03_hvg8000\stage\GBM_core_4datasets_HVG8000.main-analysis-03-streaming-seurat-v3-v2.assembled.h5ad -> G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad
[PUBLISH sequential copy] E:\glioma-cnv-scratch\main_analysis_03_hvg8000\sta

In [8]:
# =========================
# 7) Final reports and invariants
# =========================

summary_rows = [
    {
        "metric": "input_fullgene_h5ad",
        "value": str(FULLGENE_INPUT),
    },
    {
        "metric": "output_hvg_h5ad",
        "value": str(OUTPUT_HVG_H5AD),
    },
    {
        "metric": "output_hvg_compatibility_alias",
        "value": str(OUTPUT_HVG_ALIAS),
    },
    {
        "metric": "n_cells",
        "value": int(EXPECTED_OUTPUT_SHAPE[0]),
    },
    {
        "metric": "n_genes_input_after_zero_filter",
        "value": int(len(analysis_gene_names)),
    },
    {
        "metric": "n_zero_count_genes_removed",
        "value": int(n_zero_genes),
    },
    {
        "metric": "n_hvg_selected",
        "value": int(len(selected_genes)),
    },
    {
        "metric": "hvg_method_used",
        "value": hvg_method_used,
    },
    {
        "metric": "hvg_batch_key",
        "value": HVG_BATCH_KEY,
    },
    {
        "metric": "hvg_span",
        "value": HVG_SPAN,
    },
    {
        "metric": "has_counts_layer",
        "value": False,
    },
    {
        "metric": "x_contains_raw_counts",
        "value": True,
    },
    {
        "metric": "x_shape",
        "value": str(EXPECTED_OUTPUT_SHAPE),
    },
    {
        "metric": "memory_mode",
        "value": (
            "streaming_direct_hdf5_two_pass_"
            "seurat_v3_plus_chunked_output"
        ),
    },
    {
        "metric": "scvi_setup_layer_should_be",
        "value": "None",
    },
    {
        "metric": "n_core_datasets",
        "value": int(
            obs_hvg["core_dataset"].nunique()
        ),
    },
    {
        "metric": "n_samples",
        "value": int(
            obs_hvg["sample_id"].nunique()
        ),
    },
    {
        "metric": "n_patients",
        "value": int(
            obs_hvg["patient_id"].nunique()
        ),
    },
    {
        "metric": "selected_gene_sha256",
        "value": strings_sha256(
            selected_genes
        ),
    },
]

summary = pd.DataFrame(
    summary_rows
)

write_tsv_replace(
    summary,
    HVG_SUMMARY_TSV,
)

display(summary)


condition_counts = (
    obs_hvg
    .groupby(
        [
            "core_dataset",
            "condition",
        ],
        observed=True,
    )
    .size()
    .reset_index(name="n_cells")
    .sort_values(
        [
            "core_dataset",
            "condition",
        ]
    )
)

display(condition_counts)


overlap_rows = []

selected_nbatches = (
    hvg_metrics.loc[
        hvg_metrics["highly_variable"],
        "highly_variable_nbatches",
    ]
    .value_counts()
    .sort_index()
)

for nbatches, count in selected_nbatches.items():
    overlap_rows.append(
        {
            "highly_variable_nbatches": int(nbatches),
            "n_selected_genes": int(count),
        }
    )

overlap = pd.DataFrame(
    overlap_rows
)

write_tsv_replace(
    overlap,
    HVG_OVERLAP_BY_DATASET_TSV,
)

display(overlap)


if int(condition_counts["n_cells"].sum()) != EXPECTED_OUTPUT_SHAPE[0]:
    raise AssertionError(
        "Cell composition report does not sum to 351,427 cells."
    )

if len(selected_genes) != HVG_N_TOP:
    raise AssertionError(
        "HVG selection did not produce exactly 8,000 genes."
    )

if primary_validation["shape"] != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "Primary H5AD shape is incorrect."
    )

if alias_validation["shape"] != EXPECTED_OUTPUT_SHAPE:
    raise AssertionError(
        "Compatibility alias shape is incorrect."
    )

print()
print("[DONE] HVG memory-safe notebook completed successfully.")
print("Primary output:", OUTPUT_HVG_H5AD)
print("Compatibility alias:", OUTPUT_HVG_ALIAS)
print("Validated shape:", EXPECTED_OUTPUT_SHAPE)
print("Validated X encoding: csr_matrix")
print("Validated counts layer: absent by design")
print("Validated HVGs:", HVG_N_TOP)
print("scVI setup must use layer=None")


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\core_4datasets_HVG8000_summary_lowmem.tsv | shape= (19, 2)


,metric,value
0,input_fullgene_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,output_hvg_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
2,output_hvg_compatibility_alias,G:\Repository\glioma-cnv-single-cell-atlas\dat...
3,n_cells,351427
4,n_genes_input_after_zero_filter,15176
5,n_zero_count_genes_removed,0
6,n_hvg_selected,8000
7,hvg_method_used,seurat_v3_counts_batch_core_dataset_streaming_...
8,hvg_batch_key,core_dataset
9,hvg_span,0.3


,core_dataset,condition,n_cells
0,DS1_GSE103224,GBM,23207
1,DS2_GSE141383,GBM,21391
2,DS3_GSE182109,LGG,13487
3,DS3_GSE182109,ndGBM,112416
4,DS3_GSE182109,rGBM,92832
5,DS4_GSE278450,GBM,88094


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\core_4datasets_HVG8000_overlap_by_dataset_lowmem.tsv | shape= (4, 2)


,highly_variable_nbatches,n_selected_genes
0,1,1117
1,2,1309
2,3,1483
3,4,4091



[DONE] HVG memory-safe notebook completed successfully.
Primary output: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad
Compatibility alias: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_HVG8000_pre_scVI_lowmem_Xcounts.h5ad
Validated shape: (351427, 8000)
Validated X encoding: csr_matrix
Validated counts layer: absent by design
Validated HVGs: 8000
scVI setup must use layer=None


## Expected successful completion

The final cell must report:

```text
[DONE] HVG memory-safe notebook completed successfully.
Validated shape: (351427, 8000)
Validated X encoding: csr_matrix
Validated counts layer: absent by design
Validated HVGs: 8000
scVI setup must use layer=None
```

Primary output:

`data/processed/integrated/GBM_core_4datasets_HVG8000_pre_scVI_lowmem_Xcounts.h5ad`

Compatibility alias:

`data/processed/integrated/GBM_core_HVG8000_pre_scVI_lowmem_Xcounts.h5ad`

The full-gene object remains unchanged and is retained for marker DE, CNV,
and other full-feature analyses.
